# Errores que no fallan — código correcto, resultado equivocado

**Unidad 4.d** · Acompaña a `Clase_04_DatosPanel` y `Clase_05_EleccionBinaria` ·
Notas: caps. 5 y 7

Los cuatro casos de [`02_Depurar`](../02_Depurar/) eran errores de programación: código
que no hacía lo que su autor creía. Los de aquí son distintos y peores.

**El código está bien escrito, el resultado es el que la biblioteca debe devolver, y el
error está en lo que se reporta o en cómo se interpreta.** No hay nada que depurar; hay
que saber econometría.

Es la clase de error que un asistente de IA reproduce con toda naturalidad, porque su
código es correcto. La única defensa es el criterio de quien firma el resultado.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

panel = pd.read_csv("../../Clase_04_DatosPanel/wage_panel.csv")
mroz = pd.read_csv("../../Clase_05_EleccionBinaria/Mroz.csv")

print(f"Panel de salarios: {len(panel)} obs, {panel['nr'].nunique()} individuos")
print(f"Mroz:              {len(mroz)} obs")

Panel de salarios: 4360 obs, 545 individuos
Mroz:              753 obs


---
## CASO 1 — Errores estándar sin agrupar en un panel

El panel tiene 8 observaciones por individuo. Los errores de un mismo individuo están
correlacionados, de modo que MCO trata como información independiente algo que no lo es,
y **subestima la incertidumbre**.

In [2]:
formula = "lwage ~ exper + expersq + union + married + educ + black + hisp"
modelo = smf.ols(formula, data=panel).fit()
agrupado = modelo.get_robustcov_results(cov_type="cluster", groups=panel["nr"])

tabla = pd.DataFrame(
    {
        "coef": modelo.params.values,
        "ee_mco": modelo.bse.values,
        "ee_agrup": agrupado.bse,
        "t_mco": modelo.tvalues.values,
        "t_agrup": agrupado.tvalues,
    },
    index=modelo.params.index,
)
tabla["razon_ee"] = tabla["ee_agrup"] / tabla["ee_mco"]
tabla.round(4)

,coef,ee_mco,ee_agrup,t_mco,t_agrup,razon_ee
Intercept,-0.0347,0.0646,0.1201,-0.5375,-0.2890,1.8601
exper,0.0892,0.0101,0.0124,8.8200,7.1670,1.2306
expersq,-0.0028,0.0007,0.0009,-4.0272,-3.2721,1.2308
union,0.1801,0.0171,0.0276,10.5179,6.5290,1.6109
married,0.1077,0.0157,0.0261,6.8592,4.1281,1.6616
educ,0.0994,0.0047,0.0092,21.2476,10.7933,1.9686
black,-0.1438,0.0236,0.0501,-6.1055,-2.8704,2.1270
hisp,0.0157,0.0208,0.0392,0.7543,0.4005,1.8835


In [3]:
peor = tabla["razon_ee"].idxmax()
maxima = tabla["razon_ee"].max()

print(f"Los ee de MCO son hasta {maxima:.2f} veces demasiado pequeños "
      f"(variable `{peor}`).\n")

cambian = tabla[(tabla["t_mco"].abs() > 1.96) != (tabla["t_agrup"].abs() > 1.96)]
if len(cambian) == 0:
    print("En ESTOS datos ninguna conclusión se invierte: los efectos son fuertes")
    print("y sobreviven la corrección. Conviene decirlo así, sin exagerar.")
    print()
    print("Pero nótese el margen: con los ee agrupados, el umbral de significancia")
    print(f"se alcanza con un t de MCO cercano a {1.96 * maxima:.1f}. Cualquier variable")
    print(f"con t de MCO entre 2 y {1.96 * maxima:.1f} habría pasado de «significativa» a no serlo.")
else:
    print(f"Cambian de significancia al 5 %: {list(cambian.index)}")

Los ee de MCO son hasta 2.13 veces demasiado pequeños (variable `black`).

En ESTOS datos ninguna conclusión se invierte: los efectos son fuertes
y sobreviven la corrección. Conviene decirlo así, sin exagerar.

Pero nótese el margen: con los ee agrupados, el umbral de significancia
se alcanza con un t de MCO cercano a 4.2. Cualquier variable
con t de MCO entre 2 y 4.2 habría pasado de «significativa» a no serlo.


**Lo correcto:** agrupar al nivel de la unidad de asignación. Ver el capítulo 5
(inferencia en panel) y Bertrand, Duflo y Mullainathan (2004).

---
## CASO 2 — Efectos marginales en la media, en vez de promediados

En un modelo no lineal el efecto marginal **depende del punto donde se evalúa**. Hay dos
objetos distintos que se confunden a menudo:

- **AME** — se calcula el efecto de cada individuo y se promedian.
- **MEM** — se evalúa el efecto en el individuo promedio.

In [4]:
mroz["participa"] = (mroz["lfp"] == "yes").astype(int)
logit = smf.logit("participa ~ age + k5 + inc", data=mroz).fit(disp=0)

print("El «individuo promedio» de esta muestra tiene")
print(f"  {mroz['k5'].mean():.2f} hijos menores de 5 años,")
print(f"cuando el {100 * (mroz['k5'] == 0).mean():.0f} % de las mujeres no tiene ninguno.")
print("Es un individuo que no existe.")

El «individuo promedio» de esta muestra tiene
  0.24 hijos menores de 5 años,
cuando el 80 % de las mujeres no tiene ninguno.
Es un individuo que no existe.


In [5]:
ame = logit.get_margeff(at="overall").margeff
mem = logit.get_margeff(at="mean").margeff

comparacion = pd.DataFrame(
    {"AME (correcto)": ame, "MEM": mem}, index=logit.params.index[1:]
)
comparacion["error_%"] = (
    100 * (comparacion["MEM"] - comparacion["AME (correcto)"])
    / comparacion["AME (correcto)"].abs()
)
comparacion.round(4)

,AME (correcto),MEM,error_%
age,-0.0127,-0.0139,-10.0715
k5,-0.2923,-0.3217,-10.0715
inc,-0.0042,-0.0046,-10.0715


### Por qué el error es idéntico en las tres variables

No es casualidad. En un **modelo de índice único**, todo efecto marginal es el
coeficiente multiplicado por la densidad evaluada en el índice:

$$\frac{\partial P}{\partial x_j} = \lambda(\mathbf{x}'\boldsymbol{\beta}) \, \beta_j$$

Al cambiar el punto de evaluación cambia el factor $\lambda$, que es **común a todas las
variables**. De ahí que AME y MEM difieran por un mismo factor de escala y que los
cocientes entre efectos marginales sean iguales bajo ambos criterios.

**Consecuencia práctica:** la confusión no altera la comparación relativa entre
variables, pero sí toda magnitud que se reporte en términos absolutos —puntos
porcentuales de probabilidad, en este caso—.

Ninguna de las dos cifras es «un error de cálculo»: ambas son correctas para lo que cada
una mide. El error es reportar una llamándola la otra.

---
## CASO 3 — Comparar $R^2$ entre modelos con distinta variable dependiente

In [6]:
panel["wage"] = np.exp(panel["lwage"])

en_log = smf.ols("lwage ~ educ + exper", data=panel).fit()
en_nivel = smf.ols("wage ~ educ + exper", data=panel).fit()

print(f"  lwage ~ educ + exper   R^2 = {en_log.rsquared:.4f}")
print(f"  wage  ~ educ + exper   R^2 = {en_nivel.rsquared:.4f}")
print()
print("Lectura ingenua: «el modelo en logaritmos ajusta mejor».")

  lwage ~ educ + exper   R^2 = 0.1429
  wage  ~ educ + exper   R^2 = 0.1322

Lectura ingenua: «el modelo en logaritmos ajusta mejor».


Es una comparación **sin sentido**. El $R^2$ es la fracción de la varianza de *la
variable dependiente* que el modelo explica, y las dos variables dependientes son
distintas: sus varianzas no son la misma cantidad.

Un $R^2$ sólo se compara entre modelos que explican exactamente lo mismo.

Cuando hay que elegir entre nivel y logaritmo, el criterio no es el $R^2$ sino la forma
funcional que la teoría y los residuales respalden (capítulo 1, formas funcionales).

---
## CASO 4 — Leer un coeficiente log-lineal como porcentaje exacto

La variable dependiente está en logaritmos y `union` es una variable binaria.

In [7]:
modelo_log = smf.ols("lwage ~ union + educ + exper", data=panel).fit()
beta = modelo_log.params["union"]

ingenuo = 100 * beta
correcto = 100 * (np.exp(beta) - 1)

print(f"  beta_union = {beta:.4f}\n")
print(f"  Lectura habitual : {ingenuo:.2f} % más de salario")
print(f"  Lectura exacta   : {correcto:.2f} % más de salario")
print(f"  Error            : {correcto - ingenuo:.2f} puntos porcentuales")

  beta_union = 0.1777

  Lectura habitual : 17.77 % más de salario
  Lectura exacta   : 19.45 % más de salario
  Error            : 1.68 puntos porcentuales


Para una variable **discreta** el efecto porcentual exacto es $e^{\beta} - 1$, no
$\beta$. La aproximación es buena para coeficientes pequeños y se degrada rápido:

In [8]:
print(f"{'beta':>8s} {'aprox (%)':>12s} {'exacto (%)':>12s} {'error (pp)':>12s}")
for b in [0.05, 0.10, 0.18, 0.30, 0.50, 0.70]:
    ap, ex = 100 * b, 100 * (np.exp(b) - 1)
    print(f"{b:8.2f} {ap:12.2f} {ex:12.2f} {ex - ap:12.2f}")

    beta    aprox (%)   exacto (%)   error (pp)
    0.05         5.00         5.13         0.13
    0.10        10.00        10.52         0.52
    0.18        18.00        19.72         1.72
    0.30        30.00        34.99         4.99
    0.50        50.00        64.87        14.87
    0.70        70.00       101.38        31.38


Para una variable **continua** sí se interpreta como semielasticidad —cambio porcentual
ante un cambio marginal— y $\beta$ es la lectura correcta. La distinción está en el
capítulo 1, formas funcionales.

---
## Conclusión

En los cuatro casos el código es correcto y la biblioteca devolvió exactamente lo que se
le pidió. **El error está en qué se pide y en cómo se lee.** Un asistente de IA los
reproduce sin dificultad, porque no hay nada que su verificación de sintaxis pueda
detectar.

La defensa no es técnica: es saber qué objeto se necesita y qué significa. Eso es lo que
se estudia en el resto del curso.

## Tareas

1. Para cada caso, escribe la frase que **sí** sería correcta al reportar el resultado.
2. Toma un cuaderno de otra clase del repositorio y búscale alguno de los cuatro errores.
3. Pídele a un asistente de IA que estime «el efecto del sindicato sobre el salario en el
   panel» sin más indicaciones. ¿Cuál de los cuatro errores comete? ¿Los advierte?

---
Parte del curso de **Econometría I**, Facultad de Ciencias, UNAM.
Ver el [README de la carpeta](README.md) para las demás actividades.